In [1]:
import os
import json
import gc
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import optuna
import optuna.logging

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

from utils.utils import (
    connection,
    data_from_ticker,
    data_from_tpulse,
    data_from_macrofactors,
)
import utils.config_ml as config_ml

In [2]:
companies = pd.read_sql("SELECT * FROM companies", connection())
tickers = companies['ticker']

In [3]:
left_date = config_ml.LEFT_DATE
right_date = config_ml.RIGHT_DATE
train_period = config_ml.TRAIN_PERIOD
val_period = config_ml.VAL_PERIOD
test_period = config_ml.TEST_PERIOD
step = config_ml.STEP
models = config_ml.MODELS
n_trials = config_ml.N_TRIALS
metric_optuna = config_ml.METRIC_OPTUNA
top_n_features = config_ml.TOP_N_FEATURES

In [4]:
# Сбор данных
def pack_all_data_for_ml_models(ticker: str, left_date: str, right_date: str, conn):
    tpulse_data = data_from_tpulse(ticker, left_date, right_date, conn)
    ticker_data = data_from_ticker(ticker, left_date, right_date, conn)
    macrofactor_data = data_from_macrofactors(ticker, left_date, right_date, conn)

    if tpulse_data.empty or ticker_data.empty or macrofactor_data.empty:
        return pd.DataFrame()
    
    data = tpulse_data.merge(macrofactor_data, how='left', on=['dt', 'ticker'])
    data = data.merge(ticker_data, how='left', on=['dt', 'ticker'])
    
    data = data[~data['target'].isnull()].reset_index(drop=True)

    return data

In [ ]:
# Кросс-валидация скользящим окном
def sliding_windows_cross_validatin(df, train_days, val_days, test_days, step):
    n = len(df)
    min_required_days = train_days + val_days + test_days
    
    if n < min_required_days:
        return []

    windows = []
    dates_array = df['dt'].values
    current_test_end = n
    min_test_end = min_required_days
    
    while current_test_end >= min_test_end:
        test_start = current_test_end - test_days
        val_start = test_start - val_days
        train_start = val_start - train_days
        
        if train_start < 0:
            break
            
        windows.append({
            'train': df.iloc[train_start:val_start],
            'val': df.iloc[val_start:test_start],
            'test': df.iloc[test_start:current_test_end],
            'id': current_test_end,
            'dates': {
                'train': (dates_array[train_start], dates_array[val_start - 1]),
                'val': (dates_array[val_start], dates_array[test_start - 1]),
                'test': (dates_array[test_start], dates_array[current_test_end - 1])
            }
        })
        
        current_test_end -= step
        
    return windows


# Отбор топ фичей
def select_top_features(model, feature_names, top_n):
    n_features = len(feature_names)
    if top_n >= n_features:
        return feature_names

    if hasattr(model, 'coef_') and model.coef_ is not None and len(model.coef_) > 0:
        scores = np.abs(model.coef_).flatten()
    elif hasattr(model, 'feature_importances_'):
        scores = model.feature_importances_
    else:
        return feature_names[:top_n]

    partition_index = len(scores) - top_n
    top_indices_unsorted = np.argpartition(scores, partition_index)[partition_index:]
    
    top_indices = top_indices_unsorted[np.argsort(scores[top_indices_unsorted])[::-1]]
    
    return [feature_names[i] for i in top_indices]


# Подбор гиперпараметров
def make_objective(model_name, X_train, y_train, X_val, y_val):
    model_cfg = models.get(model_name)
    if not model_cfg:
        raise ValueError(f"Model configuration for '{model_name}' not found in models dict.")
        
    optuna_func = model_cfg.get('optuna_objective')
    is_catboost = (model_name == 'CatBoost')
    
    if metric_optuna == 'MAPE':
        metric_func = mean_absolute_percentage_error
    elif metric_optuna == 'MAE':
        metric_func = mean_absolute_error
    else:
        metric_func = mean_absolute_error 

    def objective(trial):
        params = {}
        
        if optuna_func and callable(optuna_func):
            params = optuna_func(trial)
            
        if is_catboost:
            model = CatBoostRegressor(**params, verbose=False)
        else:
            if callable(model_cfg['model']):
                model = model_cfg['model'](**params)
            else:
                model = model_cfg['model']
                if params:
                    model.set_params(**params)
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        return metric_func(y_val, y_pred)
        
    return objective

In [6]:
def train_val_test_ml_models(df, windows, models, ticker, left_date, right_date, train_period, val_period, test_period, step, top_n_features, metric_optuna, db_connection_func):
    print(f'Запущен цикл разработки ML моделей для прогнозирования стоимости акций "{ticker}"\n')
    
    df_ml_rows = []
    df_ml_db_rows = []
    
    db_conn = db_connection_func()
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_cols = {'dt', 'ticker', 'target'}
    feature_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    print(f'Всего числовых фичей для обучения: {len(feature_cols)}')

    for model_name in models:
        ml_info = []
        
        model_cfg = models.get(model_name)
        is_linear = (model_name == 'LinearRegression')
        is_catboost = (model_name == 'CatBoost')
        
        needs_scaling = is_linear
        
        print(f'--- Обработка модели: {model_name} ---')

        for idx, window in enumerate(windows):
            train, val, test = window['train'], window['val'], window['test']
            
            X_train, y_train = train[feature_cols], train['target']
            X_val, y_val = val[feature_cols], val['target']
            X_test, y_test = test[feature_cols], test['target']

            print(f'Окно {idx+1} | Обучение: {len(train)} дн. | Валидация: {len(val)} дн. | Тест: {len(test)} дн.')

            if model_name in ['LinearRegression', 'DecisionTree', 'GradientBoosting']:
                train = train.fillna(0)
                val = val.fillna(0)
                test = test.fillna(0)
                X_train = X_train.fillna(0)
                X_val = X_val.fillna(0)
                X_test = X_test.fillna(0)

            best_params = {}
            scaler = None
            X_train_processed = X_train
            X_test_processed = X_test

            if needs_scaling:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train)
                X_test_processed = scaler.transform(X_test)
                best_params = {}
            else:
                objective = make_objective(model_name, X_train, y_train, X_val, y_val)
                study = optuna.create_study(direction='minimize')
                study.optimize(objective, n_trials=4, show_progress_bar=False)
                best_params = study.best_params

            if is_catboost:
                model_instance = CatBoostRegressor(**best_params, verbose=False)
            else:
                model_template = model_cfg['model']
                if callable(model_template):
                    model_instance = model_template(**best_params)
                else:
                    model_instance = clone(model_template)
                    if best_params:
                        model_instance.set_params(**best_params)
            
            model_instance.fit(X_train_processed, y_train)

            selected_features = select_top_features(model_instance, feature_cols, top_n_features)
            print(f'Отобрано топ-{top_n_features} фичей, 5 лучших из них: {selected_features[:5]}')
            selected_idx = [feature_cols.index(f) for f in selected_features]

            if needs_scaling:
                X_train_final = X_train_processed[:, selected_idx]
                X_test_final = X_test_processed[:, selected_idx]
            else:
                X_train_final = X_train[selected_features]
                X_test_final = X_test[selected_features]
            
            if is_catboost:
                final_model = CatBoostRegressor(**best_params, verbose=False)
            else:
                if callable(model_template):
                    final_model = model_template(**best_params)
                else:
                    final_model = clone(model_template)
                    if best_params:
                        final_model.set_params(**best_params)
            
            final_model.fit(X_train_final, y_train)
            y_test_pred = final_model.predict(X_test_final)
            
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
            test_mae = mean_absolute_error(y_test, y_test_pred)
            print(f'Метрики лучшей модели в данном окне: MAPE={test_mape:.3f}, MAE={test_mae:.2f}\n')
            ml_info.append([final_model, selected_features, test_mape])

            del X_train, y_train, X_val, y_val, X_test, y_test, X_train_processed, X_test_processed
            del X_train_final, X_test_final, y_test_pred, model_instance, final_model
            gc.collect()

        ml_info_sorted = sorted(ml_info, key=lambda x: x[-1])
        best_model = ml_info_sorted[0][0]
        
        if ml_info:
            best_features_set = set(ml_info[0][1])
            for feature_list in ml_info[1:]:
                best_features_set.intersection_update(feature_list[1])
            best_features = list(best_features_set)
        else:
            best_features = []
            
        print('Формируется лучшая модель из построенных ранее...')
        print(f'Лучшая {model_name} модель - {best_model} с фичами в количестве {len(best_features)} шт.')
        
        mape_mean_list = []
        mae_mean_list = []

        for idx, window in enumerate(windows):
            train, test = window['train'], window['test']
            
            best_features_numeric = [f for f in best_features if f in feature_cols]
            
            if needs_scaling:
                scaler_final = StandardScaler()
                X_train_f = scaler_final.fit_transform(train[best_features_numeric])
                X_test_f = scaler_final.transform(test[best_features_numeric])
            else:
                X_train_f = train[best_features_numeric]
                X_test_f = test[best_features_numeric]
            
            y_train_f = train['target']
            y_test_f = test['target']
            
            if is_catboost:
                final_params = best_model.get_params()
                final_params['verbose'] = False
                model_eval = CatBoostRegressor(**final_params)
            else:
                model_eval = clone(best_model)
            
            model_eval.fit(X_train_f, y_train_f)
            y_pred = model_eval.predict(X_test_f)
            
            test_mape = mean_absolute_percentage_error(y_test_f, y_pred)
            test_mae = mean_absolute_error(y_test_f, y_pred)
            
            mape_mean_list.append(test_mape)
            mae_mean_list.append(test_mae)
            
            df_ml_rows.append({
                'test_period': f"{window['dates']['test'][0]} - {window['dates']['test'][1]}",
                'model_name': model_name,
                'mape': round(test_mape, 3),
                'mae': round(test_mae, 3)
            })
            
            prev_mape = ml_info[idx][-1]
            if prev_mape > 0:
                improvement_pct = round(100 * (prev_mape - test_mape) / prev_mape)
            else:
                improvement_pct = 0
                
            print(f'Окно {idx+1}: лучшая {model_name} модель имеет MAPE={test_mape:.3f} ({improvement_pct}% к модели на окне {idx+1} ранее), MAE={test_mae:.2f}')
            
            del X_train_f, X_test_f, y_pred, model_eval
            gc.collect()

        df_ml_db_rows.append({
            'dt': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'ticker': ticker,
            'left_date': left_date,
            'right_date': right_date,
            'model_name': model_name,
            'model_value': str(best_model),
            'train_period': train_period,
            'val_period': val_period,
            'test_period': test_period,
            'step': step,
            'mape_mean': float(round(np.mean(mape_mean_list), 3)),
            'mae_mean': float(round(np.mean(mae_mean_list), 3)),
            'best_features': json.dumps(best_features)
        })
        
        print('------------------------------------------------------------------------------------------------------------')

    if df_ml_rows:
        df_ml = pd.DataFrame(df_ml_rows, columns=['test_period', 'model_name', 'mape', 'mae'])
    else:
        df_ml = pd.DataFrame(columns=['test_period', 'model_name', 'mape', 'mae'])
        
    if df_ml_db_rows:
        df_ml_db = pd.DataFrame(df_ml_db_rows, columns=['dt', 'ticker', 'left_date', 'right_date', 'model_name', 'model_value', 'train_period', 'val_period', 'test_period', 'step', 'mape_mean', 'mae_mean', 'best_features'])
    else:
        df_ml_db = pd.DataFrame(columns=['dt', 'ticker', 'left_date', 'right_date', 'model_name', 'model_value', 'train_period', 'val_period', 'test_period', 'step', 'mape_mean', 'mae_mean', 'best_features'])

    if not df_ml_db.empty:
        df_ml_db.to_sql('ml_models_data', con=db_conn, if_exists='append', index=False)

    return df_ml.sort_values(by=['test_period', 'mae']), df_ml_db

In [7]:
for ticker in tickers:
    try:
        data = pack_all_data_for_ml_models(ticker, left_date, right_date, connection())

        df = data[~data['target'].isnull()].reset_index(drop=True)

        if df.empty:
            print(f'Нет данных для обучения модели {ticker}')
            df_ml = pd.DataFrame(columns=['test_period', 'model_name', 'mape', 'mae'])
            df_ml_db = pd.DataFrame(columns=['dt', 'ticker', 'left_date', 'right_date', 'model_name', 'model_value', 'train_period', 'val_period', 'test_period', 'step', 'mape_mean', 'mae_mean', 'best_features'])
        else:
            windows = sliding_windows_cross_validatin(df, train_period, val_period, test_period, step)
            
            if not windows:
                print(f'Недостаточно данных для кросс-валидации {ticker}')
                df_ml = pd.DataFrame(columns=['test_period', 'model_name', 'mape', 'mae'])
                df_ml_db = pd.DataFrame(columns=['dt', 'ticker', 'left_date', 'right_date', 'model_name', 'model_value', 'train_period', 'val_period', 'test_period', 'step', 'mape_mean', 'mae_mean', 'best_features'])
            else:
                df_ml, df_ml_db = train_val_test_ml_models(
                    df, 
                    windows, 
                    models, 
                    ticker,
                    left_date,
                    right_date,
                    train_period,
                    val_period,
                    test_period,
                    step,
                    top_n_features,
                    metric_optuna,
                    connection
                )

        print(df_ml)
        print(df_ml_db)

        db_conn = connection()

        if not df.empty:
            df_summary = df[['dt', 'ticker', 'target']].copy()
            df_summary['left_date'] = left_date
            df_summary['right_date'] = right_date
            df_summary['created_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            df_summary.to_sql('ml_features_summary', con=db_conn, if_exists='append', index=False)
            print(f'Сохранена сводка данных: {len(df_summary)} строк')

        windows_log = []
        for idx, window in enumerate(windows):
            windows_log.append({
                'ticker': ticker,
                'window_id': idx + 1,
                'train_start': window['dates']['train'][0],
                'train_end': window['dates']['train'][1],
                'val_start': window['dates']['val'][0],
                'val_end': window['dates']['val'][1],
                'test_start': window['dates']['test'][0],
                'test_end': window['dates']['test'][1],
                'train_size': len(window['train']),
                'val_size': len(window['val']),
                'test_size': len(window['test']),
                'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })

        if windows_log:
            df_windows = pd.DataFrame(windows_log)
            df_windows.to_sql('ml_windows_log', con=db_conn, if_exists='append', index=False)
            print(f'Сохранено {len(df_windows)} окон кросс-валидации')

        df_ml_with_info = df_ml.copy()
        df_ml_with_info['ticker'] = ticker
        df_ml_with_info['left_date'] = left_date
        df_ml_with_info['right_date'] = right_date
        df_ml_with_info['created_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        df_ml_with_info.to_sql('ml_results_detailed', con=db_conn, if_exists='append', index=False)
        print(f'Сохранено {len(df_ml_with_info)} результатов тестирования')

        print(f'Все данные сохранены в БД для тикера {ticker}')

        del data, df, df_summary, df_ml_with_info, df_windows
        if 'windows_log' in locals():
            del windows_log

        gc.collect()
        
    except Exception as e:
        print(e)
        continue

Запущен цикл разработки ML моделей для прогнозирования стоимости акций "SBER"

Всего числовых фичей для обучения: 7619
--- Обработка модели: DecisionTree ---
Окно 1 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['RUSFAR1WRT_open_lag_5', 'low_rm_2', 'RUSFAR1MRT_close_lag_2', 'high_lag_16_low_lag_16_diff_pct', 'RUCBTRNS_open_rm_19']
Метрики лучшей модели в данном окне: MAPE=0.003, MAE=1.03



/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train = train.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val = val.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To 

Окно 2 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['tfidf_результат_sum', 'boll_lb_lag_22', 'RUCBTRNS_open_lag_21', 'RUCBTRNS_open_lag_15', 'RUCBTRNS_open_rm_15']


/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train = train.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val = val.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To 

Метрики лучшей модели в данном окне: MAPE=0.015, MAE=4.75

Формируется лучшая модель из построенных ранее...
Лучшая DecisionTree модель - DecisionTreeRegressor(max_depth=11, min_samples_leaf=4, min_samples_split=4,
                      random_state=42) с фичами в количестве 491 шт.
Окно 1: лучшая DecisionTree модель имеет MAPE=0.004 (-22% к модели на окне 1 ранее), MAE=1.26
Окно 2: лучшая DecisionTree модель имеет MAPE=0.020 (-35% к модели на окне 2 ранее), MAE=6.38
------------------------------------------------------------------------------------------------------------
--- Обработка модели: RandomForest ---
Окно 1 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['high', 'low', 'RUABITR_open_lag_14', 'RUBMI_open_lag_10', 'MOEXINN_close_rm_2']
Метрики лучшей модели в данном окне: MAPE=0.012, MAE=3.84

Окно 2 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['high', 'high_rm_21', 'low_rm_8', 'h

...

Формируется лучшая модель из построенных ранее...
Лучшая DecisionTree модель - DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, min_samples_split=3,
                      random_state=42) с фичами в количестве 500 шт.
Окно 1: лучшая DecisionTree модель имеет MAPE=0.048 (0% к модели на окне 1 ранее), MAE=3.60
------------------------------------------------------------------------------------------------------------
--- Обработка модели: RandomForest ---
Окно 1 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['low_rm_2', 'low', 'high', 'rsi_14_lag_22', 'IMOEX_open_lag_6']
Метрики лучшей модели в данном окне: MAPE=0.049, MAE=3.68

Формируется лучшая модель из построенных ранее...
Лучшая RandomForest модель - RandomForestRegressor(bootstrap=False, max_depth=12,
                      max_features=0.7184113214793384, min_samples_leaf=8,
                      min_samples_split=14, n_estimators=133, n_jobs=-1,
                      random_st

/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train = train.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val = val.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To 

Формируется лучшая модель из построенных ранее...
Лучшая DecisionTree модель - DecisionTreeRegressor(max_depth=3, min_samples_split=8, random_state=42) с фичами в количестве 500 шт.
Окно 1: лучшая DecisionTree модель имеет MAPE=0.062 (-10% к модели на окне 1 ранее), MAE=0.07
------------------------------------------------------------------------------------------------------------
--- Обработка модели: RandomForest ---
Окно 1 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['low', 'volume_rm_13', 'high_lag_1', 'low_rm_2', 'high_rm_2']
Метрики лучшей модели в данном окне: MAPE=0.062, MAE=0.07

Формируется лучшая модель из построенных ранее...
Лучшая RandomForest модель - RandomForestRegressor(bootstrap=False, max_depth=12,
                      max_features=0.6492527054819095, min_samples_leaf=3,
                      min_samples_split=4, n_estimators=215, n_jobs=-1,
                      random_state=42) с фичами в количестве 500 шт.
Окно 

/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train = train.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  val = val.fillna(0)
/var/folders/3y/q4zxlyrs3md__fhhs8mw34qc0000gn/T/ipykernel_31251/1128830820.py:38: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To 

Формируется лучшая модель из построенных ранее...
Лучшая DecisionTree модель - DecisionTreeRegressor(max_depth=14, min_samples_leaf=6, min_samples_split=9,
                      random_state=42) с фичами в количестве 500 шт.
Окно 1: лучшая DecisionTree модель имеет MAPE=0.106 (0% к модели на окне 1 ранее), MAE=5.50
------------------------------------------------------------------------------------------------------------
--- Обработка модели: RandomForest ---
Окно 1 | Обучение: 21 дн. | Валидация: 14 дн. | Тест: 7 дн.
Отобрано топ-500 фичей, 5 лучших из них: ['low', 'high', 'high_rm_2', 'low_rm_3', 'low_rm_2']
Метрики лучшей модели в данном окне: MAPE=0.098, MAE=5.08

Формируется лучшая модель из построенных ранее...
Лучшая RandomForest модель - RandomForestRegressor(bootstrap=False, max_depth=16,
                      max_features=0.37512173834416473, min_samples_leaf=6,
                      min_samples_split=6, n_estimators=242, n_jobs=-1,
                      random_state=42) с ф